# De reseñas de clientes en varios idiomas a un tablero de decisión estratégica

- Artículo completo en la plataforma: https://fuzzyfrog.ai/es/ai-lab/proyectos/negocios/analisis-sentimiento-resenas-clientes-balanced-scorecard/
- Este notebook reconstruye, con reseñas sintéticas, el mismo flujo aplicado en el proyecto original: limpieza, detección de idioma, traducción, análisis de sentimiento y descubrimiento de temas sin etiquetas previas.
- **Nota:** el dataset usado es sintético, generado para fines demostrativos, con un restaurante y reseñas ficticias. No contiene datos reales de la fuente original.

## Diagrama de arquitectura

`Reseñas multiidioma exportadas → limpieza y detección de idioma → traducción a inglés → tokenización con stopwords por idioma → (a) análisis de sentimiento y (b) clustering no supervisado → mapeo de resultados a KPIs, validado con el negocio → tablero de decisión`

Este notebook cubre desde la limpieza hasta el análisis. El mapeo final a KPIs se hace junto con el equipo del negocio, no es un paso automático.

## Instalación de dependencias

Se usan las mismas herramientas del proyecto original: detección de idioma, traducción, análisis de sentimiento léxico y clustering.

In [ ]:
!pip install langdetect textblob --quiet
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('vader_lexicon')

In [ ]:
import pandas as pd
import numpy as np
from langdetect import detect
from textblob import TextBlob
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.cluster import KMeans
import nltk

## Carga de datos

Se carga el dataset sintético de reseñas multiidioma. En el proyecto original este paso incluía además reconstruir encabezados desalineados y eliminar columnas vacías de la exportación en Excel; sobre el dataset sintético ese trabajo ya no es necesario porque se generó limpio a propósito.

In [ ]:
df = pd.read_csv("outputs/dataset_sintetico_resenas_multiidioma.csv")
df.head()

## Explicación de datos

- `comment`: texto de la reseña, en el idioma original.
- `language`: idioma reportado por el dataset sintético (en el proyecto real este campo se detecta automáticamente).
- `year`: año de la reseña, usado más adelante para ver la tendencia en el tiempo.
- `reference_polarity`: etiqueta de referencia, solo para poder evaluar el pipeline sobre datos sintéticos. En producción esta columna no existe, se calcula.

In [ ]:
print(df.shape)
df["language"].value_counts()

## Análisis de datos / EDA

Distribución de reseñas por año, para detectar si el volumen de opiniones cambia con el tiempo.

In [ ]:
df.groupby("year")["review_id"].count()

## Modelado: detección de idioma y traducción

Se detecta el idioma de cada reseña de forma automática y se traduce a inglés antes de analizar el sentimiento, igual que en el pipeline original. El analizador de polaridad usado rinde mejor en ese idioma que evaluando texto multiidioma sin normalizar.

In [ ]:
def detect_language_safe(text):
    try:
        return detect(text)
    except Exception:
        return "unknown"

def translate_to_english(text, lang):
    if lang == "en":
        return text
    try:
        return str(TextBlob(text).translate(from_lang=lang, to="en"))
    except Exception:
        return text

df["detected_language"] = df["comment"].apply(detect_language_safe)
df["translated_comment"] = df.apply(
    lambda row: translate_to_english(row["comment"], row["detected_language"]), axis=1
)
df[["comment", "detected_language", "translated_comment"]].head()

## Modelado: análisis de sentimiento

Se calcula la polaridad de cada reseña ya traducida con un analizador léxico, y se etiqueta como positiva, negativa o neutra, igual que en el proyecto original.

In [ ]:
sid = SentimentIntensityAnalyzer()

def label_polarity(text):
    score = sid.polarity_scores(text)["compound"]
    if score > 0.05:
        return "positive"
    elif score < -0.05:
        return "negative"
    return "neutral"

df["predicted_polarity"] = df["translated_comment"].apply(label_polarity)
df["predicted_polarity"].value_counts()

## Evaluación

Se compara la polaridad predicha contra la etiqueta de referencia del dataset sintético, solo como sanity check del pipeline de sentimiento.

In [ ]:
accuracy = (df["reference_polarity"] == df["predicted_polarity"]).mean()
print(f"Coincidencia con la etiqueta de referencia (dataset sintético): {accuracy:.1%}")

## Modelado: matriz de frecuencia de términos y clustering

Se tokeniza cada reseña, se eliminan las palabras vacías del idioma correspondiente, y se construye una matriz de frecuencia de términos. Sobre esa matriz se aplica clustering no supervisado, igual que en el proyecto original, para descubrir temas sin etiquetas previas.

In [ ]:
stopwords_by_language = {
    "en": set(stopwords.words("english")),
    "es": set(stopwords.words("spanish")),
    "it": set(stopwords.words("italian")),
    "de": set(stopwords.words("german")),
    "fr": set(stopwords.words("french")),
}

def tokenize_comment(text, lang):
    tokenizer = nltk.RegexpTokenizer(r"\w+")
    tokens = tokenizer.tokenize(text.lower())
    stop_words = stopwords_by_language.get(lang, set())
    return [t for t in tokens if t not in stop_words]

df["tokens"] = df.apply(lambda row: tokenize_comment(row["comment"], row["language"]), axis=1)
df[["comment", "tokens"]].head()

In [ ]:
all_tokens = sorted(set(token for tokens in df["tokens"] for token in tokens))
print("Tokens unicos:", len(all_tokens))

frequency_matrix = []
for tokens in df["tokens"]:
    row_counts = [tokens.count(term) for term in all_tokens]
    frequency_matrix.append(row_counts)

frequency_matrix = np.array(frequency_matrix)
frequency_matrix.shape

In [ ]:
kmeans = KMeans(init="k-means++", n_clusters=3, n_init=10, random_state=42)
kmeans.fit(frequency_matrix)
df["cluster"] = kmeans.predict(frequency_matrix)
df["cluster"].value_counts()

## Interpretación de los clusters

Se revisan las palabras más frecuentes de cada grupo, para entender qué tema domina cada uno. Este es el punto donde, en el proyecto real, se valida el significado de cada cluster junto con el negocio antes de mapearlo a un indicador.

In [ ]:
from collections import Counter

for cluster_id in sorted(df["cluster"].unique()):
    tokens_in_cluster = [t for tokens in df[df["cluster"] == cluster_id]["tokens"] for t in tokens]
    top_words = Counter(tokens_in_cluster).most_common(8)
    print(f"Cluster {cluster_id}: {top_words}")

## Celda final de texto: hallazgos principales

- Traducir antes de clasificar mejora la confiabilidad del análisis de sentimiento en un dataset multiidioma, frente a analizar cada idioma con reglas propias.
- El clustering no supervisado permite descubrir agrupaciones temáticas sin necesitar etiquetas previas, pero el significado de negocio de cada grupo no es automático, requiere validación de quien conoce el negocio.
- La detección automática de idioma puede fallar en textos muy cortos o sin estructura, ese tipo de casos necesita revisión manual, no una regla más en el código.
- Este notebook cubre limpieza, sentimiento y descubrimiento de temas. El paso de mapear resultados a indicadores de negocio se hace en conjunto con el cliente, fuera del código.

## Nota de actualización

Este flujo se construyó el mismo año en que se lanzó ChatGPT, antes de que existieran modelos de lenguaje con la capacidad actual. Se conserva aquí tal cual, como registro honesto de cómo se resolvió en su momento.

Hoy, buena parte de este pipeline podría simplificarse: un modelo de lenguaje actual puede leer una reseña en cualquier idioma sin traducirla primero, asignarle sentimiento y tema en un mismo paso, y hasta sugerir a qué indicador de negocio corresponde. La validación humana seguiría siendo necesaria, pero varios de los pasos separados de este notebook podrían resolverse con muchas menos piezas.